In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
def get_losses(lines):
    losses = []
    for line in lines:
        loss = float(line.split('training loss: ')[1].split('\n')[0])
        losses.append(loss)
    return np.array(losses)

def get_scores(lines):
    scores = []
    for line in lines:
        score = float(line.split('score: ')[1].split('\n')[0])
        scores.append(score)
    return np.array(scores)

def get_lr(lines):
    d = dict()
    for line in lines:
        lr = float(line.split('to ')[1].split('.\n')[0])
        epoch = int(line.split('Epoch ')[1].split(':')[0])
        d[epoch] = lr
    return d

def plot_log_losses(losses):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.arange(50, 50*len(losses), 50), y=np.log(losses), mode='lines'))

    fig.update_layout(
        title='Training loss - log scale',
        xaxis_title='Iteration',
        yaxis_title='Log(loss)'
    )
    fig.write_image("../figures/logloss.png")
    fig.show()

def plot_losses(losses):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.arange(50, 50*len(losses), 50), y=losses, mode='lines'))

    fig.update_layout(
        title='Training loss',
        xaxis_title='Iteration',
        yaxis_title='Loss'
    )
    fig.write_image("../figures/loss.png")
    fig.show()

def plot_scores(scores, learning_rates):
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=np.arange(1, 1+len(scores)), y=scores, mode='lines', name='Score'))
    for epoch, lr in learning_rates.items():
        fig.add_trace(go.Scatter(x=[epoch], y=[scores[epoch-1]], mode='markers', name=f'lr set to {lr}', marker=dict(color='red', size=5)))
    
    fig.update_layout(
        title='Validation score',
        xaxis_title='Epoch',
        yaxis_title='Score'
    )

    fig.write_image("../figures/scores.png")
    fig.show()

In [3]:
n_epochs = 60
filename = f'../logs/2024-01-27_13:48:26_scibert_scivocab_cased_32_bs_{n_epochs}_epochs.log'
with open(filename, 'r') as f:
    lines = f.readlines()
    loss_lines = [line for line in lines if 'Iteration' in line]
    scores_lines = [line for line in lines if 'score: ' in line]
    lr_lines = [line for line in lines if 'learning rate of group 0' in line]
    losses = get_losses(loss_lines)
    scores = get_scores(scores_lines)
    learning_rates = get_lr(lr_lines)

In [4]:
plot_losses(losses)
plot_log_losses(losses)
plot_scores(scores, learning_rates)